# MLflow & database

Тестирование подключения к MLFLow и БД из Jupyter Notebook

## Configuration

In [1]:
# Хак, чтобы добавить корень проекта в path для импорта модулей
import sys
from pathlib import Path

def find_repo_root() -> Path:
    path = Path.cwd().resolve()
    for candidate in [path, *path.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Could not find repo root (pyproject.toml). Open this notebook from the repo.")

sys.path.append(str(find_repo_root()))
sys.path

['/home/fiberfox/.local/share/uv/python/cpython-3.12.7-linux-x86_64-gnu/lib/python312.zip',
 '/home/fiberfox/.local/share/uv/python/cpython-3.12.7-linux-x86_64-gnu/lib/python3.12',
 '/home/fiberfox/.local/share/uv/python/cpython-3.12.7-linux-x86_64-gnu/lib/python3.12/lib-dynload',
 '',
 '/home/fiberfox/Projects/HSEAIMag2025/stocks-advisor/.venv/lib/python3.12/site-packages',
 '/home/fiberfox/Projects/HSEAIMag2025/stocks-advisor']

In [3]:
# Код для загрузки конфига и подключения к MLFlow
# Для работы с продом в корне проекта должен быть .env файл с всеми нужными переменными окружения
# Для локальной разработки .env не нужен
from jupyter_utils import setup_jupyter_notebook

# Подключение к проду
setup_jupyter_notebook(environment='prod', experiment='test')

# Подключение к локальным БД и MLFlow
# setup_jupyter_notebook(environment='local', experiment='test')

Environment: prod
APP_CONFIG: config.toml
Tracking URI: http://localhost:5050
S3 endpoint: http://localhost:9050
Experiment: test
Database: localhost:15432/stocks_advisor_db


## MLFlow

Пример логгирования эксперимента в MLFlow

In [4]:
from datetime import UTC, datetime
import mlflow

run_name = f"smoke-test-{datetime.now(UTC).strftime('%Y%m%d-%H%M%S')}"

with mlflow.start_run(run_name=run_name) as run:
    mlflow.log_param("source", "mlflow_smoke_test")
    mlflow.log_metric("ping", 1.0)
    mlflow.log_dict({"status": "ok"}, "smoke.json")

print(f"Run ID: {run.info.run_id}")
print(f"Run name: {run_name}")
print("Smoke test passed.")

🏃 View run smoke-test-20260531-135453 at: http://localhost:5050/#/experiments/2/runs/a121855d9f634e23b534208dac809ef5
🧪 View experiment at: http://localhost:5050/#/experiments/2
Run ID: a121855d9f634e23b534208dac809ef5
Run name: smoke-test-20260531-135453
Smoke test passed.


## Database queries

Примеры загрузки данных из PostgreSQL:

- Стоимость акций
- Данные по новостям: тикеры, сектор, sentiment

In [5]:
# Загрузка свечей MOEX

from app.core.database import AssetCandleRepository, get_db_session

async with get_db_session() as session:
    repo = AssetCandleRepository(session)
    candles_by_ticker = await repo.get_dataframe_by_ticker(limit=1000)

print(f'{len(candles_by_ticker)} tickers')
for ticker, df in candles_by_ticker.items():
    print(f'  {ticker}: {len(df)} rows')

11 tickers
  BRENT: 80 rows
  CNYRUBF: 80 rows
  EURRUBF: 76 rows
  GAZP: 116 rows
  GLDRUB_TOM: 50 rows
  IMOEX: 55 rows
  LKOH: 116 rows
  ROSN: 116 rows
  SBER: 116 rows
  T: 115 rows
  USDRUBF: 80 rows


In [6]:
# Генерация фич

from app.core.processors.feature_generator import FeatureGenerator


features_by_ticker = {}
fg = FeatureGenerator()

for ticker, df in candles_by_ticker.items():
    features_by_ticker[ticker] = fg.process(
        df=df,
        include_original=False,
        add_targets=False,
        clean=True,
    )
features_by_ticker['SBER'].head()

IndexError: index 199 is out of bounds for axis 0 with size 25

In [ ]:
# Загрузка данных по новостям

from app.core.database import NewsArticleRepository, get_db_session

LIMIT = 100

async with get_db_session() as session:
    repo = NewsArticleRepository(session)
    enrichments_df = await repo.get_all_enrichments_as_dataframe(limit=LIMIT, ticker='SBER', sector='MOEXFN')

print(f'Enrichments: {len(enrichments_df)} rows')
enrichments_df.head()

Enrichments: 100 rows


,id,news_article_id,published_at,tickers,sector,sentiment,sentiment_score
0,49186,134601,2024-11-25 18:36:00,[SBER],MOEXFN,negative,0.937763
1,49191,134598,2024-11-25 19:05:00,"[VTBR, ROSN, LKOH, GAZP, SBER, TATN]",MOEXFN,negative,0.872153
2,49210,134585,2024-11-25 21:54:00,"[SBER, VTBR, TATN]",MOEXFN,negative,0.540392
3,49298,135456,2024-11-26 10:16:00,"[SBER, VTBR, TATN, LKOH]",MOEXFN,negative,0.928977
4,49317,135441,2024-11-26 12:39:00,[SBER],MOEXFN,neutral,0.577350
